# Encoder A: features, data split, model, training, clean EER

PHC-351 · Hiding a Voice · Bhavishay (24324008), IIT Roorkee

Code files 01_features.py, 02_dataset.py, 03_model_A.py, 04_train_A.py, 05_eer_A.py from `code/`, as Colab cells in run order.

**How to run:** Runtime → Change runtime type → **T4 GPU** → Save, then **Runtime → Run all**. Allow Google Drive access when asked.

No Google Drive needed. **04 trains encoder A (8 epochs): this is the long part.** The model is saved in the run folder, not on Drive.

**Changes from the original cells, so that the notebook runs on its own:** a setup cell at the top; cells marked `ADDED for the re-run` (names the original session defined in cells that were not kept, and two result printouts); and file paths, marked `# re-run:`, so that everything is written into a run folder in this Colab session and the files on Drive are never overwritten. Everything else is the code as run; `code/` keeps the original paths.

## Setup (added)

In [ ]:
# ===== SETUP (added so this notebook runs top to bottom from GitHub; not part of the original runs) =====
import os, glob, shutil, torch, torchaudio, numpy as np, soundfile as sf
RUN = '/content/PHC351_rerun'          # this session's working folder; Colab deletes it when the session ends
os.makedirs(RUN, exist_ok=True)

# Audio files: newer torchaudio needs the extra package torchcodec for load/save.
# If it is not installed, load/save go through soundfile instead (same samples, same shapes).
try:
    torchaudio.save(f'{RUN}/_probe.wav', torch.zeros(1, 160), 16000); torchaudio.load(f'{RUN}/_probe.wav')
except Exception:
    def _load(path, *a, **k):
        x, sr = sf.read(path, dtype='float32', always_2d=True)
        return torch.from_numpy(np.ascontiguousarray(x.T)), sr
    def _save(path, x, sr, *a, **k):
        sf.write(path, np.ascontiguousarray(x.detach().cpu().numpy().T), sr, subtype='FLOAT')
    torchaudio.load, torchaudio.save = _load, _save
    print('audio load/save: soundfile (torchcodec not installed)')

# LibriSpeech train-clean-100 (about 6 GB, 5-15 minutes; skipped if already downloaded in this session)
os.chdir('/content')
if not os.path.exists('/content/LibriSpeech/train-clean-100'):
    print('downloading LibriSpeech train-clean-100 ...')
    os.system("curl -sL -C - -o tc100.tar.gz "
              "https://www.openslr.org/resources/12/train-clean-100.tar.gz && tar -xzf tc100.tar.gz")
_n = len(glob.glob('/content/LibriSpeech/train-clean-100/*/*/*.flac'))
print('LibriSpeech files:', _n, '(expected about 28539)')
assert _n > 0, 'LibriSpeech download failed: run this cell again'

## 01_features.py

First log-mel feature test on one LibriSpeech file (with a plot).

In [ ]:
import torch, torchaudio, glob, matplotlib.pyplot as plt

# ---- settings (your pipeline constants) ----
SR, N_FFT, HOP, WIN, N_MELS = 16000, 512, 160, 400, 80

# ---- 1. load one file ----
path = glob.glob('/content/LibriSpeech/train-clean-100/*/*/*.flac')[0]
wav, sr = torchaudio.load(path)          # wav: (channels, samples)
wav = wav.mean(0)                        # mono: (samples,)
print("file:", path.split('/')[-1], "| sr:", sr, "| samples:", wav.shape[0],
      "| seconds:", round(wav.shape[0]/sr, 2))

# ---- 2. STFT: frames -> complex spectrum ----
window = torch.hann_window(WIN)
X = torch.stft(wav, n_fft=N_FFT, hop_length=HOP, win_length=WIN,
               window=window, center=True, return_complex=True)
print("STFT:", tuple(X.shape), "= (bins, frames)")

# ---- 3. magnitude -> power ----
power = X.abs() ** 2                     # phase discarded here
print("power:", tuple(power.shape))

# ---- 4. mel filterbank: 257 bins -> 80 bands ----
fb = torchaudio.functional.melscale_fbanks(
        n_freqs=N_FFT // 2 + 1, f_min=0.0, f_max=SR / 2,
        n_mels=N_MELS, sample_rate=SR, norm=None, mel_scale="htk")
print("filterbank:", tuple(fb.shape), "= (bins, bands)")
mel = fb.T @ power                       # (80, frames)

# ---- 5. log ----
logmel = torch.log(mel + 1e-6)
print("log-mel:", tuple(logmel.shape),
      "| min/max:", round(logmel.min().item(),2), round(logmel.max().item(),2))

# ---- 6. look at it ----
fig, ax = plt.subplots(2, 1, figsize=(10, 6))
ax[0].plot(wav.numpy()); ax[0].set_title("waveform")
ax[1].imshow(logmel.numpy(), origin='lower', aspect='auto', cmap='magma')
ax[1].set_title("log-mel"); ax[1].set_xlabel("frame"); ax[1].set_ylabel("mel band")
plt.tight_layout(); plt.show()

## 02_dataset.py

Log-mel front end, the speaker-disjoint split (200 / 51 speakers, seed 0), 3-second crops, the data loader.

In [ ]:
import torch, torchaudio, glob, random
from torch.utils.data import Dataset, DataLoader

SR, N_FFT, HOP, WIN, N_MELS = 16000, 512, 160, 400, 80
CHUNK = 3 * SR                                   # 48000 samples

_window = torch.hann_window(WIN)
_fb = torchaudio.functional.melscale_fbanks(
        n_freqs=N_FFT//2+1, f_min=0.0, f_max=SR/2,
        n_mels=N_MELS, sample_rate=SR, norm=None, mel_scale="htk")

def logmel(wav):
    """wav: (samples,) -> (80, frames)"""
    X = torch.stft(wav, n_fft=N_FFT, hop_length=HOP, win_length=WIN,
                   window=_window.to(wav.device), center=True, return_complex=True)
    return torch.log(_fb.T.to(wav.device) @ (X.abs()**2) + 1e-6)

# ---------- collect files and split BY SPEAKER ----------
files = sorted(glob.glob('/content/LibriSpeech/train-clean-100/*/*/*.flac'))
pairs = [(p, p.split('/')[-3]) for p in files]

speakers = sorted({s for _, s in pairs})
random.Random(0).shuffle(speakers)
n_train = int(0.8 * len(speakers))
train_spk, held_spk = set(speakers[:n_train]), set(speakers[n_train:])
spk2idx = {s: i for i, s in enumerate(sorted(train_spk))}

print(f"speakers: {len(speakers)} -> train {len(train_spk)}, held-out {len(held_spk)}")

class SpeakerSet(Dataset):
    def __init__(self, pairs, keep, labelled=True):
        self.items = [(p, s) for p, s in pairs if s in keep]
        self.labelled = labelled
    def __len__(self):
        return len(self.items)
    def __getitem__(self, i):
        path, spk = self.items[i]
        wav, _ = torchaudio.load(path)
        wav = wav.mean(0)
        if wav.shape[0] < CHUNK:                       # pad short files
            wav = torch.nn.functional.pad(wav, (0, CHUNK - wav.shape[0]))
        else:                                          # random 3s crop
            start = random.randint(0, wav.shape[0] - CHUNK)
            wav = wav[start:start + CHUNK]
        label = spk2idx[spk] if self.labelled else -1
        return logmel(wav), label

train_ds = SpeakerSet(pairs, train_spk)
held_ds  = SpeakerSet(pairs, held_spk, labelled=False)
print("train clips:", len(train_ds), "| held-out clips:", len(held_ds))

train_dl = DataLoader(train_ds, batch_size=32, shuffle=True,
                      num_workers=2, drop_last=True)

xb, yb = next(iter(train_dl))
print("batch features:", tuple(xb.shape), "| labels:", tuple(yb.shape),
      "| label range:", yb.min().item(), "-", yb.max().item())

## 03_model_A.py

Encoder A (SpeakerNet): 4 conv blocks, statistics pooling, 192-number fingerprint.

In [ ]:
import torch, torch.nn as nn, torch.nn.functional as F

class SpeakerNet(nn.Module):
    def __init__(self, n_classes, emb_dim=192):
        super().__init__()
        def blk(cin, cout, stride):
            return nn.Sequential(
                nn.Conv2d(cin, cout, 3, stride=stride, padding=1),
                nn.BatchNorm2d(cout), nn.ReLU(inplace=True))
        # stride (freq, time): shrink frequency, keep time resolution
        self.conv = nn.Sequential(
            blk(1,   32, (2,1)),     # 80 -> 40 mel rows
            blk(32,  64, (2,1)),     # 40 -> 20
            blk(64, 128, (2,1)),     # 20 -> 10
            blk(128,128, (2,1)),     # 10 -> 5
        )
        self.proj = nn.Linear(128*5*2, emb_dim)   # *5 freq rows, *2 for mean+std
        self.bn   = nn.BatchNorm1d(emb_dim)
        self.head = nn.Linear(emb_dim, n_classes)

    def forward(self, x):                 # x: (B, 80, T)
        x = x.unsqueeze(1)                # (B, 1, 80, T)
        h = self.conv(x)                  # (B, 128, 5, T)
        B, C, Fq, T = h.shape
        h = h.reshape(B, C*Fq, T)         # (B, 640, T)  frequency folded in
        mu  = h.mean(dim=2)
        sd  = h.var(dim=2, unbiased=False).clamp_min(1e-8).sqrt()
        pooled = torch.cat([mu, sd], dim=1)        # (B, 1280)
        e = self.bn(self.proj(pooled))             # (B, 192) embedding
        return e, self.head(e)

dev = 'cuda' if torch.cuda.is_available() else 'cpu'
model = SpeakerNet(n_classes=len(spk2idx)).to(dev)
print("params (M):", round(sum(p.numel() for p in model.parameters())/1e6, 2))

e, logits = model(xb.to(dev))
print("embedding:", tuple(e.shape), "| logits:", tuple(logits.shape))

## 04_train_A.py

Training encoder A: Adam 1e-3, cross-entropy, 8 passes, saved to Drive.

In [ ]:
import time, torch.nn as nn

opt = torch.optim.Adam(model.parameters(), lr=1e-3)
lossf = nn.CrossEntropyLoss()
EPOCHS = 8

for ep in range(EPOCHS):
    model.train(); t0 = time.time()
    tot = correct = 0; running = 0.0
    for xb, yb in train_dl:
        xb, yb = xb.to(dev), yb.to(dev)
        _, logits = model(xb)
        loss = lossf(logits, yb)
        opt.zero_grad(); loss.backward(); opt.step()

        running += loss.item() * yb.size(0)
        correct += (logits.argmax(1) == yb).sum().item()
        tot     += yb.size(0)

    print(f"epoch {ep+1}/{EPOCHS} | loss {running/tot:.3f} "
          f"| train acc {100*correct/tot:.1f}% | {time.time()-t0:.0f}s")
    torch.save(model.state_dict(), f'{RUN}/modelA.pt')   # re-run: saved in the run folder (original: '/content/drive/MyDrive/modelA.pt')

## 05_eer_A.py

Equal error rate of encoder A on the 51 held-out speakers (clean audio): 0.92%, tau 0.463.

In [ ]:
import torch, random, numpy as np

model.eval()

# build embeddings for held-out speakers
by_spk = {}
for p, s in pairs:
    if s in held_spk:
        by_spk.setdefault(s, []).append(p)
by_spk = {s: v[:10] for s, v in by_spk.items() if len(v) >= 10}   # 10 clips each
print("held-out speakers used:", len(by_spk))

@torch.no_grad()
def embed(path):
    wav, _ = torchaudio.load(path); wav = wav.mean(0)
    if wav.shape[0] < CHUNK:
        wav = torch.nn.functional.pad(wav, (0, CHUNK - wav.shape[0]))
    wav = wav[:CHUNK]
    e, _ = model(logmel(wav).unsqueeze(0).to(dev))
    return torch.nn.functional.normalize(e, dim=1)[0].cpu()

emb = {s: [embed(p) for p in v] for s, v in by_spk.items()}
print("embeddings done")

# same-speaker and different-speaker pairs
rng = random.Random(0)
same, diff = [], []
spk_list = list(emb)
for s in spk_list:
    for i in range(len(emb[s])):
        for j in range(i+1, len(emb[s])):
            same.append(float(emb[s][i] @ emb[s][j]))
for _ in range(len(same)):
    a, b = rng.sample(spk_list, 2)
    diff.append(float(rng.choice(emb[a]) @ rng.choice(emb[b])))

same, diff = np.array(same), np.array(diff)
print(f"pairs: {len(same)} same, {len(diff)} different")
print(f"mean similarity -> same {same.mean():.3f} | different {diff.mean():.3f}")

# EER
ths = np.linspace(-1, 1, 2001)
far = [(diff > t).mean() for t in ths]      # strangers accepted
frr = [(same <= t).mean() for t in ths]     # genuine rejected
i = int(np.argmin(np.abs(np.array(far) - np.array(frr))))
print(f"EER = {100*(far[i]+frr[i])/2:.2f}%  at threshold {ths[i]:.3f}")